# ⚙️ Technical Documentation — Pump Failure Prediction MVP

## 1. Data and Model Pipeline

```
[Raw CSV Files — Kaggle]
         │
         ▼
┌─────────────────────────────────┐
│  src/data/loader.py             │
│  • Read 4 CSV files             │
│  • Add label column             │
│  • Merge into single DataFrame  │
│  • Save: data/processed/        │
│    dataset_merged.csv           │
└──────────────┬──────────────────┘
               │
               ▼
┌─────────────────────────────────┐
│  src/data/preprocessor.py       │
│  • Drop irrelevant columns      │
│  • Drop redundant columns       │
│  • Handle missing values        │
│  • Clip outliers (Z>3σ)         │
│  • Stratified split (70/15/15)  │
│  • StandardScaler (fit=train)   │
│  • Borderline-SMOTE (train)     │
│  • Save: models/scaler.joblib   │
└──────────────┬──────────────────┘
               │
               ▼
┌─────────────────────────────────┐
│  src/features/engineer.py       │
│  • 3 ratio features             │
│  • 24 rolling features          │
│  • 3 delta features             │
│  • VarianceThreshold filter     │
│  → 37 final features            │
└──────────────┬──────────────────┘
               │
               ▼
┌─────────────────────────────────┐
│  src/models/trainer.py          │
│  • Optuna tuning (50 trials)    │
│  • XGBoost training             │
│  • Random Forest (baseline)     │
│  • 5-fold CV evaluation         │
│  • MLflow logging               │
│  • Model Registry registration  │
└──────────────┬──────────────────┘
               │
               ▼
┌─────────────────────────────────┐
│  src/models/evaluator.py        │
│  • Load model from Registry     │
│  • Evaluate on test set         │
│  • ROC curves per class         │
│  • Normalized confusion matrix  │
│  • Log to MLflow                │
└──────────────┬──────────────────┘
               │
               ▼
┌─────────────────────────────────┐
│  src/models/persistor.py        │
│  • Export model → .joblib       │
│  • Validate artifacts           │
│  • Save model_metadata.json     │
└──────────────┬──────────────────┘
               │
               ▼
┌─────────────────────────────────┐
│  app/Home.py (Streamlit)        │
│  • Upload CSV (7 raw features)  │
│  • Feature Engineering          │
│  • Scaler transform             │
│  • XGBoost inference            │
│  • Dashboard visualization      │
└─────────────────────────────────┘
```

## 2. Project Structure

```
pump-failure-prediction/
│
├── data/
│   ├── raw/                        # Original CSVs from Kaggle
│   │   ├── dane_OT.csv
│   │   ├── dane_ut1.csv
│   │   ├── dane_ut2.csv
│   │   └── dane_ut3.csv
│   ├── processed/                  # Merged and preprocessed data
│   │   ├── dataset_merged.csv
│   │   └── sample_input.csv        # Test file for dashboard
│   └── features/                   # Engineered features (optional cache)
│
├── src/
│   ├── __init__.py
│   ├── data/
│   │   ├── __init__.py
│   │   ├── loader.py               # Etapa 1 — Ingestion
│   │   └── preprocessor.py         # Etapa 3 — Preprocessing
│   ├── features/
│   │   ├── __init__.py
│   │   └── engineer.py             # Etapa 4 — Feature Engineering
│   └── models/
│       ├── __init__.py
│       ├── trainer.py              # Etapa 5 — Training + MLflow
│       ├── evaluator.py            # Etapa 6 — Evaluation
│       └── persistor.py            # Etapa 7 — Persistence
│
├── models/                         # Saved artifacts
│   ├── scaler.joblib
│   ├── xgboost_model.joblib
│   └── model_metadata.json
│
├── app/                            # Streamlit dashboard
│   ├── Home.py
│   ├── assets/
│   │   └── test_bench_diagram.png
│   └── pages/
│       └── 01_model_explanation.py
│
├── notebooks/                      # Exploration notebooks
│   ├── 01_eda.ipynb
│   └── 04_modeling.ipynb
│
├── reports/                        # Generated plots
│   ├── class_distribution.png
│   ├── correlation_matrix.png
│   ├── confusion_matrix_XGBoost_Tuned.png
│   ├── confusion_matrix_test_XGBoost_Tuned.png
│   ├── feature_importance_XGBoost_Tuned.png
│   └── roc_curves_XGBoost_Tuned.png
│
├── docs/                           # Documentation
│   ├── 01_business_documentation.md
│   ├── 02_data_documentation.md
│   ├── 03_model_documentation.md
│   └── 04_technical_documentation.md
│
├── mlruns/                         # MLflow filesystem (legacy)
├── mlflow.db                       # MLflow SQLite backend
├── requirements.txt
└── README.md
```

---

## 3. Inference Pipeline (Dashboard)

```
Input: CSV with 7 raw sensor columns
         │
         ├── Validate columns
         ├── Fill NaN with median
         ├── StandardScaler.transform() (7 features)
         ├── Feature Engineering (→ 37 features)
         ├── Reorder to match training feature order
         └── XGBClassifier.predict() + predict_proba()
         │
Output: predicted_class + confidence + prob per class
```

> ⚠️ **Critical:** Scaler must be applied BEFORE feature engineering,
> because it was fitted on the 7 raw features in training.